# 03 — Modelagem
Comparação de modelos com cross-validation, tuning do melhor e salvamento do pipeline final.

In [ ]:
import sys
sys.path.append('..')

import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_log_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

from src.feature_engineering import create_features, NAN_MEANS_NONE, ORDINAL_QUAL_COLS

RANDOM_STATE = 42
CV = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 1. Preparação dos dados

In [ ]:
train_raw = pd.read_csv('../data/treino.csv')

# Remove os 2 outliers identificados no EDA
outlier_mask = (train_raw['GrLivArea'] > 4000) & (train_raw['SalePrice'] < 300_000)
train_raw = train_raw[~outlier_mask].copy()
print(f'Treino após remover outliers: {train_raw.shape}')

y = train_raw['SalePrice'].copy()
X = create_features(train_raw.drop(columns=['SalePrice']))

print(f'X shape: {X.shape}')
print(f'y range: {y.min():,.0f} — {y.max():,.0f}')

## 2. Definindo o preprocessor

In [ ]:
with open('../src/col_config.json') as f:
    cfg = json.load(f)

num_cols       = cfg['num_cols']
cat_none_cols  = cfg['cat_none_cols']
cat_real_cols  = cfg['cat_real_cols']
already_encoded = cfg['already_encoded']

def build_preprocessor():
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ])
    cat_none_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
        ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    cat_real_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    ord_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
    ])
    return ColumnTransformer([
        ('num',      num_pipe,      num_cols),
        ('cat_none', cat_none_pipe, cat_none_cols),
        ('cat_real', cat_real_pipe, cat_real_cols),
        ('ordinal',  ord_pipe,      already_encoded),
    ], remainder='drop')

print('Configuração de colunas carregada.')

## 3. Comparação de modelos via cross-validation

`TransformedTargetRegressor` aplica `log1p` no target dentro do CV → sem leakage.

In [ ]:
def rmsle(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)  # garante sem negativos
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

def make_pipeline(model):
    """Cria pipeline completo: preprocessor + TransformedTargetRegressor."""
    reg = TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1,
    )
    return Pipeline([
        ('prep', build_preprocessor()),
        ('model', reg),
    ])

models = {
    'LinearRegression': LinearRegression(),
    'Ridge':            Ridge(alpha=10, random_state=RANDOM_STATE),
    'Lasso':            Lasso(alpha=0.001, max_iter=10_000, random_state=RANDOM_STATE),
    'RandomForest':     RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost':          XGBRegressor(
                            n_estimators=500, learning_rate=0.05,
                            max_depth=4, subsample=0.8, colsample_bytree=0.8,
                            random_state=RANDOM_STATE, verbosity=0, n_jobs=-1
                        ),
}

results = {}
for name, model in models.items():
    pipe = make_pipeline(model)
    # sklearn scoring: neg_mean_squared_log_error (RMSLE = sqrt(-score))
    scores = cross_val_score(
        pipe, X, y, cv=CV,
        scoring='neg_mean_squared_log_error',
        n_jobs=-1
    )
    rmsle_scores = np.sqrt(-scores)
    results[name] = {
        'RMSLE_mean': rmsle_scores.mean(),
        'RMSLE_std':  rmsle_scores.std(),
    }
    print(f'{name:20s}  RMSLE={rmsle_scores.mean():.4f} ± {rmsle_scores.std():.4f}')

results_df = pd.DataFrame(results).T.sort_values('RMSLE_mean')
print('\n--- Ranking ---')
print(results_df.round(4))

In [ ]:
# Gráfico comparativo
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(
    results_df.index,
    results_df['RMSLE_mean'],
    xerr=results_df['RMSLE_std'],
    color='steelblue', alpha=0.8, capsize=4
)
ax.set_xlabel('RMSLE (menor = melhor)')
ax.set_title('Cross-validation RMSLE por modelo (5-fold)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Tuning do melhor modelo (XGBoost)

Ajuste fino com `RandomizedSearchCV`.

In [ ]:
from scipy.stats import randint, uniform

xgb_pipe = make_pipeline(
    XGBRegressor(random_state=RANDOM_STATE, verbosity=0, n_jobs=-1)
)

param_dist = {
    'model__regressor__n_estimators':   randint(300, 800),
    'model__regressor__max_depth':       randint(3, 7),
    'model__regressor__learning_rate':   uniform(0.01, 0.1),
    'model__regressor__subsample':       uniform(0.6, 0.4),
    'model__regressor__colsample_bytree': uniform(0.5, 0.5),
    'model__regressor__min_child_weight': randint(1, 6),
    'model__regressor__reg_alpha':        uniform(0, 1),
    'model__regressor__reg_lambda':       uniform(0.5, 2),
}

search = RandomizedSearchCV(
    xgb_pipe,
    param_distributions=param_dist,
    n_iter=30,
    scoring='neg_mean_squared_log_error',
    cv=CV,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

search.fit(X, y)

best_rmsle = np.sqrt(-search.best_score_)
print(f'\nMelhor RMSLE (CV): {best_rmsle:.4f}')
print(f'Melhores parâmetros:')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

## 5. Avaliação final — R², RMSLE e MAE em dólares

In [ ]:
from sklearn.model_selection import cross_val_predict

best_pipe = search.best_estimator_

# Predições OOF (out-of-fold) para métricas realistas
y_oof = cross_val_predict(best_pipe, X, y, cv=CV, n_jobs=-1)
y_oof = np.maximum(y_oof, 0)

r2   = r2_score(y, y_oof)
rmsle_val = rmsle(y, y_oof)
mae  = mean_absolute_error(y, y_oof)

print(f'R²    : {r2:.4f}')
print(f'RMSLE : {rmsle_val:.4f}')
print(f'MAE   : ${mae:,.0f}')

# Resíduos
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_oof, y, alpha=0.3, s=10, color='steelblue')
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=1)
axes[0].set_xlabel('Predito')
axes[0].set_ylabel('Real')
axes[0].set_title('Predito vs Real (OOF)')

residuals = y - y_oof
axes[1].scatter(y_oof, residuals, alpha=0.3, s=10, color='coral')
axes[1].axhline(0, color='k', lw=1)
axes[1].set_xlabel('Predito')
axes[1].set_ylabel('Resíduo')
axes[1].set_title('Resíduos')

plt.tight_layout()
plt.show()

## 6. Salvando o modelo final

In [ ]:
# Re-treina no dataset completo (sem hold-out) para maximizar a capacidade preditiva
best_pipe.fit(X, y)

joblib.dump(best_pipe, '../modelo.pkl')
print('Modelo salvo em modelo.pkl')

# Verificação rápida
loaded = joblib.load('../modelo.pkl')
sample_pred = loaded.predict(X.head(3))
print(f'Predição de teste (3 amostras): {sample_pred.round(0)}')
print(f'Valores reais                 : {y.head(3).values}')

In [ ]:
# Feature importance do XGBoost (top 20)
import re

xgb_model = best_pipe.named_steps['model'].regressor_
prep       = best_pipe.named_steps['prep']

# Recupera nomes das features após OHE
try:
    ohe_none_feats = list(prep.named_transformers_['cat_none'].named_steps['ohe'].get_feature_names_out(cat_none_cols))
    ohe_real_feats = list(prep.named_transformers_['cat_real'].named_steps['ohe'].get_feature_names_out(cat_real_cols))
    all_feat_names = num_cols + ohe_none_feats + ohe_real_feats + already_encoded
    importances = pd.Series(xgb_model.feature_importances_, index=all_feat_names)
    top20 = importances.sort_values(ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(8, 6))
    top20.plot.barh(ax=ax, color='steelblue')
    ax.invert_yaxis()
    ax.set_title('Top 20 Feature Importances — XGBoost')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Não foi possível recuperar feature names: {e}')
    print('XGBoost feature importance (por índice):')
    print(pd.Series(xgb_model.feature_importances_).sort_values(ascending=False).head(20))